## 2. spaCy Pipeline Integration

The `whitakers_words` component plugs into any LatinCy pipeline, providing
dictionary lookups via `token._.lexicon` and rule-based parses via `token._.ww`,
ranked by POS agreement and multi-signal disambiguation.

In [1]:
from pathlib import Path

PROJECT = Path("..").resolve()
LEXICON_PATH = PROJECT / "data" / "json" / "lexicon.json"
ANALYZER_PATH = PROJECT / "data" / "json" / "analyzer.json"

print(f"Lexicon: {LEXICON_PATH.name} ({LEXICON_PATH.stat().st_size / 1e6:.1f} MB)")
print(f"Analyzer: {ANALYZER_PATH.name} ({ANALYZER_PATH.stat().st_size / 1e6:.1f} MB)")

Lexicon: lexicon.json (17.0 MB)
Analyzer: analyzer.json (15.9 MB)


## 1. Standalone Analyzer

The `Analyzer` replicates Whitaker's Words core algorithm: for each Latin form, try every possible stem+ending split, match stems against DICTLINE (39,600+ entries) and endings against INFLECTS (1,785+ patterns). Returns all grammatically valid parses. `sum/esse` and its compounds are fully supported — the original Ada program hardcoded these forms, so we patch them into the database at build time (`db/patches.py`).

In [2]:
from latincy_lexicon.analyzer import Analyzer

analyzer = Analyzer.from_json(str(ANALYZER_PATH))

print("Analyzer loaded.")

Analyzer loaded.


In [3]:
# Analyze a single word
form = "fecerunt"
parses = analyzer.analyze(form)

print(f"{form!r} → {len(parses)} parse(s)\n")
for i, p in enumerate(parses, 1):
    d = p.to_dict()
    print(f"  [{i}] {d['lemma']} ({d['pos']})")
    feats = {k: d[k] for k in ('tense', 'voice', 'mood', 'person', 'number', 'case', 'gender') if k in d}
    if feats:
        print(f"      {feats}")
    print(f"      stem={d['stem']!r} + ending={d['ending']!r}")
    print(f"      {d['meaning'][:80]}")
    print()

'fecerunt' → 1 parse(s)

  [1] facio (V)
      {'tense': 'PERF', 'voice': 'ACTIVE', 'mood': 'IND', 'person': '3', 'number': 'P'}
      stem='fec' + ending='erunt'
      make/build/construct/create/cause/do; have built/made; fashion; work (metal);



In [4]:
# Ambiguous forms — where WW shines
ambiguous = ["est", "genus", "oris", "arma", "ducis", "nova", "venit"]

for form in ambiguous:
    parses = analyzer.analyze(form)
    print(f"{form!r} → {len(parses)} parse(s)")
    for p in parses[:3]:  # top 3 by frequency
        d = p.to_dict()
        # `dict.get(k, '')` only kicks in when the key is missing — coalesce
        # explicit None values to '' for verb parses where case/number are unset.
        case = d.get("case") or ""
        number = d.get("number") or ""
        print(f"  {d['lemma']:15s} {d['pos']:5s} {case:3s}.{number:1s}  {d['meaning'][:50]}")
    if len(parses) > 3:
        print(f"  ... +{len(parses)-3} more")
    print()

'est' → 3 parse(s)
  sum             V        .S  be; exist; (also used to form verb perfect passive
  sum             V        .S  be; exist; (also used to form verb perfect passive
  edo             V        .S  eat/consume/devour; eat away (fire/water/disease);

'genus' → 14 parse(s)
  genu            N     NOM.S  knee;
  genu            N     VOC.S  knee;
  genu            N     GEN.S  knee;
  ... +11 more

'oris' → 9 parse(s)
  os              N     GEN.S  mouth, speech, expression; face; pronunciation;
  os              N     NOM.P  mouth, speech, expression; face; pronunciation;
  os              N     ACC.P  mouth, speech, expression; face; pronunciation;
  ... +6 more

'arma' → 7 parse(s)
  armum           N     NOM.P  arms (pl.), weapons, armor, shield; close fighting
  armum           N     VOC.P  arms (pl.), weapons, armor, shield; close fighting
  armum           N     ACC.P  arms (pl.), weapons, armor, shield; close fighting
  ... +4 more

'ducis' → 5 parse(s)
  dux      

In [5]:
# Enclitic handling: -que, -ne, -ve
for form in ["armaque", "estne", "patriave"]:
    parses = analyzer.analyze(form)
    print(f"{form!r} → {len(parses)} parse(s)")
    for p in parses[:2]:
        d = p.to_dict()
        print(f"  {d['lemma']:15s} {d['pos']:5s}  {d['meaning'][:60]}")
    print()

'armaque' → 7 parse(s)
  armum           N      arms (pl.), weapons, armor, shield; close fighting weapons; 
  armum           N      arms (pl.), weapons, armor, shield; close fighting weapons; 

'estne' → 3 parse(s)
  sum             V      be; exist; (also used to form verb perfect passive tenses) w
  sum             V      be; exist; (also used to form verb perfect passive tenses) w

'patriave' → 13 parse(s)
  patria          N      native land; home, native city; one's country;
  patria          N      native land; home, native city; one's country;



## 2. spaCy Pipeline Integration

The `whitakers_words` component plugs into any LatinCy pipeline, providing
dictionary lookups via `token._.lexicon` and rule-based parses via `token._.ww`,
ranked by POS agreement and multi-signal disambiguation.

In [6]:
import spacy
import latincy_lexicon.spacy  # registers factories

nlp = spacy.load("la_core_web_lg")

# Add Whitaker's Words after the full pipeline (POS, morph, lemma available for ranking)
nlp.add_pipe("whitakers_words", config={
    "lexicon_path": str(LEXICON_PATH),
    "analyzer_path": str(ANALYZER_PATH),
}, last=True)

print("Pipeline:", nlp.pipe_names)

Pipeline: ['tok2vec', 'senter', 'normer', 'tagger', 'morphologizer', 'trainable_lemmatizer', 'lookup_lemmatizer', 'uv_normalizer', 'parser', 'harmonizer', 'remorpher', 'ner', 'whitakers_words']


In [7]:
# Process the opening of the Aeneid
text = "Arma virumque cano, Troiae qui primus ab oris Italiam fato profugus Laviniaque venit litora."
doc = nlp(text)

print(f"{'TOKEN':12s} {'LEMMA':12s} {'POS':5s} {'LEXICON':7s} TOP GLOSS")
print("-" * 85)
for token in doc:
    if token.is_punct or token.is_space:
        continue
    lex = token._.lexicon
    ww = token._.ww
    has_lex = "yes" if lex else "--"

    # Top gloss from lexicon
    gloss = ""
    if lex:
        gloss = lex[0].get("glosses", [""])[0][:40] if isinstance(lex[0].get("glosses"), list) else str(lex[0].get("glosses", ""))[:40]
    elif ww:
        gloss = ww[0].get("meaning", "")[:40]

    print(f"{token.text:12s} {token.lemma_:12s} {token.pos_:5s} {has_lex:7s} {gloss}")

TOKEN        LEMMA        POS   LEXICON TOP GLOSS
-------------------------------------------------------------------------------------
Arma         arma         NOUN  yes     arms (pl.), weapons, armor, shield
uirum        uir          NOUN  yes     man
que          que          CCONJ yes     -que = and (enclitic, translated before 
cano         cano         VERB  yes     sing, celebrate, chant
Troiae       Troia        PROPN --      
qui          qui          PRON  yes     who
primus       primus       ADJ   yes     first, foremost/best, chief, principal
ab           ab           ADP   yes     by (agent), from (departure, cause, remo
oris         ora          NOUN  yes     shore, coast
Italiam      Italia       PROPN yes     Italy
fato         fatum        NOUN  yes     utterance, oracle
profugus     profugus     ADJ   yes     fugitive/fleeing/escaping
Lauinia      lauinia      ADJ   --      
que          que          CCONJ yes     -que = and (enclitic, translated before 
uenit      

In [8]:
# Deep dive: what does WW say about "cano"?
token = [t for t in doc if t.text.lower() == "cano"][0]

print(f"Token: {token.text!r}")
print(f"spaCy says: lemma={token.lemma_!r}, POS={token.pos_}, morph={token.morph}")
print()
print("WW analyses (ranked by POS match + frequency):")
if token._.ww:
    for i, p in enumerate(token._.ww[:5], 1):
        feats = {k: p[k] for k in ('tense', 'voice', 'mood', 'person', 'number', 'case', 'gender') if k in p}
        print(f"  [{i}] {p['lemma']:15s} {p['pos']:5s} {feats}")
        print(f"      {p['meaning'][:70]}")

Token: 'cano'
spaCy says: lemma='cano', POS=VERB, morph=Aspect=Imp|Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbForm=Fin|Voice=Act

WW analyses (ranked by POS match + frequency):
  [1] cano            V     {'tense': 'PRES', 'voice': 'ACTIVE', 'mood': 'IND', 'person': '1', 'number': 'S'}
      sing, celebrate, chant; crow; recite; play (music)/sound (horn); foret
  [2] canus           ADJ   {'number': 'S', 'case': 'DAT', 'gender': 'M'}
      white, gray; aged, old, wise; hoary; foamy, white-capped; white w/snow
  [3] canus           ADJ   {'number': 'S', 'case': 'ABL', 'gender': 'M'}
      white, gray; aged, old, wise; hoary; foamy, white-capped; white w/snow
  [4] canus           ADJ   {'number': 'S', 'case': 'DAT', 'gender': 'N'}
      white, gray; aged, old, wise; hoary; foamy, white-capped; white w/snow
  [5] canus           ADJ   {'number': 'S', 'case': 'ABL', 'gender': 'N'}
      white, gray; aged, old, wise; hoary; foamy, white-capped; white w/snow


## 3. Cross-Validation: Statistical vs Rule-Based

The real power: find where spaCy and Words disagree. Disagreements are either model errors
(spaCy got it wrong) or ambiguities worth flagging.

In [9]:
passages = {
    "Aeneid I.1": "Arma virumque cano, Troiae qui primus ab oris Italiam fato profugus Laviniaque venit litora.",
    "Caesar BG I.1": "Gallia est omnis divisa in partes tres, quarum unam incolunt Belgae, aliam Aquitani, tertiam qui ipsorum lingua Celtae, nostra Galli appellantur.",
    "Cicero Cat. I.1": "Quo usque tandem abutere, Catilina, patientia nostra? Quam diu etiam furor iste tuus nos eludet?",
    "Catullus 5": "Vivamus mea Lesbia atque amemus, rumoresque senum severiorum omnes unius aestimemus assis.",
    "Ovid Met. I.1": "In nova fert animus mutatas dicere formas corpora.",
}

from latincy_lexicon.align.normalize import normalize_latin

total_tokens = 0
ww_covered = 0
lemma_agree = 0
disagreements = []

for title, passage in passages.items():
    doc = nlp(passage)
    for token in doc:
        if token.is_punct or token.is_space:
            continue
        total_tokens += 1
        if token._.ww:
            ww_covered += 1
            top_lemma = normalize_latin(token._.ww[0]["lemma"])
            spacy_lemma = normalize_latin(token.lemma_)
            if top_lemma == spacy_lemma or analyzer.lemmas_equivalent(top_lemma, spacy_lemma):
                lemma_agree += 1
            else:
                disagreements.append({
                    "passage": title,
                    "token": token.text,
                    "spacy_lemma": token.lemma_,
                    "spacy_pos": token.pos_,
                    "ww_lemma": token._.ww[0]["lemma"],
                    "ww_pos": token._.ww[0]["pos"],
                    "ww_meaning": token._.ww[0].get("meaning", "")[:50],
                })

print(f"Coverage: {ww_covered}/{total_tokens} tokens analyzed ({ww_covered/total_tokens*100:.0f}%)")
print(f"Lemma agreement: {lemma_agree}/{ww_covered} ({lemma_agree/ww_covered*100:.0f}%)")
print(f"Disagreements: {len(disagreements)}")

Coverage: 70/73 tokens analyzed (96%)
Lemma agreement: 68/70 (97%)
Disagreements: 2


In [10]:
# Show disagreements — these are the interesting cases
if disagreements:
    print(f"{'PASSAGE':15s} {'TOKEN':12s} {'spaCy':15s} {'WW':15s} WW GLOSS")
    print("-" * 90)
    for d in disagreements:
        spacy_col = f"{d['spacy_lemma']} ({d['spacy_pos']})"
        ww_col = f"{d['ww_lemma']} ({d['ww_pos']})"
        print(f"{d['passage']:15s} {d['token']:12s} {spacy_col:15s} {ww_col:15s} {d['ww_meaning']}")
else:
    print("No disagreements!")

PASSAGE         TOKEN        spaCy           WW              WW GLOSS
------------------------------------------------------------------------------------------
Caesar BG I.1   Celtae       Celtae (PROPN)  celtus (ADJ)    Celts; (inhabitants of central Gaul);
Catullus 5      Lesbia       Lesbia (PROPN)  lesbium (N)     Lesbian wine, wine from the island of Lesbos; vess


## 4. Lexicon as a Gloss Source

The `lexicon_lookup` component provides dictionary-quality glosses for every lemma in WW —
useful for reading aids, vocabulary tools, and hover-over annotations.

In [11]:
# Interlinear gloss for Caesar
text = "Gallia est omnis divisa in partes tres."
doc = nlp(text)

print("INTERLINEAR GLOSS")
print("=" * 60)
print()
for token in doc:
    if token.is_punct or token.is_space:
        continue
    lex = token._.lexicon
    if lex:
        meaning = lex[0].get("glosses", [""])
        if isinstance(meaning, list):
            meaning = meaning[0] if meaning else ""
        meaning = str(meaning)[:50]
    else:
        meaning = "(?)"
    print(f"  {token.text:12s}  {meaning}")

INTERLINEAR GLOSS

  Gallia        Gaul
  est           be
  omnis         each, every, every one (of a number)
  diuisa        divide
  in            in, on, at (space)
  partes        part, region
  tres          three


## 5. Paradigm Generator

The `Generator` is the inverse of the Analyzer: given a lemma, it produces all inflected
forms with UD morphological features. Same data source (`analyzer.json`), opposite direction.

In [12]:
from latincy_lexicon.generator import Generator

gen = Generator.from_json(str(ANALYZER_PATH))

# Generate the full paradigm of "amo" (to love)
forms = gen.generate("amo", pos="V")
print(f"amo → {len(forms)} forms\n")

# Show present indicative active
print("Present Indicative Active:")
for f in forms:
    feats = dict(kv.split("=") for kv in f.feats.split("|") if "=" in kv)
    if (feats.get("Mood") == "Ind" and feats.get("Tense") == "Pres"
            and feats.get("Voice") == "Act" and feats.get("VerbForm") == "Fin"):
        print(f"  {f.form:15} {f.feats}")

amo → 277 forms

Present Indicative Active:
  amo             Aspect=Imp|Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbForm=Fin|Voice=Act
  amas            Aspect=Imp|Mood=Ind|Number=Sing|Person=2|Tense=Pres|VerbForm=Fin|Voice=Act
  amat            Aspect=Imp|Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin|Voice=Act
  amamus          Aspect=Imp|Mood=Ind|Number=Plur|Person=1|Tense=Pres|VerbForm=Fin|Voice=Act
  amatis          Aspect=Imp|Mood=Ind|Number=Plur|Person=2|Tense=Pres|VerbForm=Fin|Voice=Act
  amant           Aspect=Imp|Mood=Ind|Number=Plur|Person=3|Tense=Pres|VerbForm=Fin|Voice=Act


In [13]:
# Noun paradigm: rex (king), 3rd declension — paradigm-sorted
forms = gen.generate("rex", pos="N", sort="paradigm")
print(f"rex → {len(forms)} forms\n")

for f in forms:
    print(f"  {f.form:15} {f.feats}")

rex → 12 forms

  rex             Case=Nom|Gender=Masc|Number=Sing
  regis           Case=Gen|Gender=Masc|Number=Sing
  regi            Case=Dat|Gender=Masc|Number=Sing
  rex             Case=Voc|Gender=Masc|Number=Sing
  regibus         Case=Dat|Gender=Masc|Number=Plur
  regibus         Case=Abl|Gender=Masc|Number=Plur
  regem           Case=Acc|Gender=Com|Number=Sing
  rege            Case=Abl|Gender=Com|Number=Sing
  reges           Case=Nom|Gender=Com|Number=Plur
  regum           Case=Gen|Gender=Com|Number=Plur
  reges           Case=Acc|Gender=Com|Number=Plur
  reges           Case=Voc|Gender=Com|Number=Plur


In [14]:
# Irregular verb: sum (to be) — suppletive stems (s-, es-, fu-, fo-)
forms = gen.generate("sum")
print(f"sum → {len(forms)} forms (UPOS: {forms[0].upos})\n")

# Show present indicative
print("Present Indicative:")
for f in forms:
    feats = dict(kv.split("=") for kv in f.feats.split("|") if "=" in kv)
    if (feats.get("Mood") == "Ind" and feats.get("Tense") == "Pres"
            and feats.get("VerbForm") == "Fin"):
        print(f"  {f.form:15} {f.feats}")

# Show imperfect subjunctive (essem/forem doublets)
print("\nImperfect Subjunctive:")
for f in forms:
    feats = dict(kv.split("=") for kv in f.feats.split("|") if "=" in kv)
    if (feats.get("Mood") == "Sub" and feats.get("Tense") == "Imp"
            and feats.get("VerbForm") == "Fin"):
        print(f"  {f.form:15} {f.feats}")

sum → 143 forms (UPOS: AUX)

Present Indicative:
  sum             Aspect=Imp|Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbForm=Fin|Voice=Act
  es              Aspect=Imp|Mood=Ind|Number=Sing|Person=2|Tense=Pres|VerbForm=Fin|Voice=Act
  est             Aspect=Imp|Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin|Voice=Act
  sumus           Aspect=Imp|Mood=Ind|Number=Plur|Person=1|Tense=Pres|VerbForm=Fin|Voice=Act
  estis           Aspect=Imp|Mood=Ind|Number=Plur|Person=2|Tense=Pres|VerbForm=Fin|Voice=Act
  sunt            Aspect=Imp|Mood=Ind|Number=Plur|Person=3|Tense=Pres|VerbForm=Fin|Voice=Act

Imperfect Subjunctive:
  essem           Aspect=Imp|Mood=Sub|Number=Sing|Person=1|Tense=Imp|VerbForm=Fin|Voice=Act
  esses           Aspect=Imp|Mood=Sub|Number=Sing|Person=2|Tense=Imp|VerbForm=Fin|Voice=Act
  esset           Aspect=Imp|Mood=Sub|Number=Sing|Person=3|Tense=Imp|VerbForm=Fin|Voice=Act
  essemus         Aspect=Imp|Mood=Sub|Number=Plur|Person=1|Tense=Imp|VerbForm=Fin|Voice=Act
 

In [15]:
# Build form→lemma lookup tables (useful for downstream NLP pipelines)
# Iterate via generate(sort="paradigm") to get pedagogical order rather
# than the alphabetic sort that to_lookup_dict + sorted() would produce.
print("Form → lemma (paradigm order)\n")
for lemma in ["puella", "rex"]:
    print(f"{lemma}:")
    seen = set()
    for f in gen.generate(lemma, pos="N", sort="paradigm"):
        if f.form not in seen:
            seen.add(f.form)
            print(f"  {f.form:20} → {f.lemma}")
    print()

Form → lemma (paradigm order)

puella:
  puella               → puella
  puellae              → puella
  puellam              → puella
  puellarum            → puella
  puellis              → puella
  puellas              → puella

rex:
  rex                  → rex
  regis                → rex
  regi                 → rex
  regibus              → rex
  regem                → rex
  rege                 → rex
  reges                → rex
  regum                → rex



## 6. Reinflection with `paradigm_generator`

The `paradigm_generator` spaCy component attaches the full paradigm to each token
and provides `token._.reinflect()` — change one or more morphological features
and get back the correct surface form.

In [16]:
# Load a fresh pipeline with the paradigm generator
import spacy
nlp2 = spacy.load("la_core_web_lg")
nlp2.add_pipe("paradigm_generator", config={
    "analyzer_path": str(ANALYZER_PATH),
}, last=True)

doc = nlp2("Roma clara est. Poeta bonus carmina pulchra scribit.")

print(f"{'TOKEN':12} {'LEMMA':10} {'POS':6} FORMS")
print("-" * 50)
for token in doc:
    if token.is_punct or token.is_space:
        continue
    n = len(token._.paradigm) if token._.paradigm else 0
    print(f"{token.text:12} {token.lemma_:10} {token.pos_:6} {n}")

TOKEN        LEMMA      POS    FORMS
--------------------------------------------------
Roma         Roma       PROPN  14
clara        clarus     ADJ    84
est          sum        AUX    143
Poeta        poeta      NOUN   12
bonus        bonus      ADJ    84
carmina      carmen     NOUN   15
pulchra      pulcher    ADJ    84
scribit      scribo     VERB   260


In [17]:
# Reinflection: change morphological features, get the right Latin form
tokens = {t.text: t for t in doc if not t.is_punct and not t.is_space}

est = tokens["est"]
bonus = tokens["bonus"]
scribit = tokens["scribit"]
clara = tokens["clara"]

print("Reinflection examples:")
print(f"  est      → plural:      {est._.reinflect(Number='Plur')}")
print(f"  bonus    → feminine:    {bonus._.reinflect(Gender='Fem')}")
print(f"  bonus    → superlative: {bonus._.reinflect(Degree='Sup')}")
print(f"  clara    → plural:      {clara._.reinflect(Number='Plur')}")
print(f"  scribit  → plural:      {scribit._.reinflect(Number='Plur')}")
print(f"  scribit  → imperfect:   {scribit._.reinflect(Tense='Imp')}")
print(f"  scribit  → passive:     {scribit._.reinflect(Voice='Pass')}")

Reinflection examples:
  est      → plural:      sunt
  bonus    → feminine:    bona
  bonus    → superlative: optimus
  clara    → plural:      clarae
  scribit  → plural:      scribunt
  scribit  → imperfect:   scribebat
  scribit  → passive:     scribitur
